## 1. Imports

In [ ]:
%pip install opencv-python numpy scipy

In [1]:
import cv2
import numpy as np
from scipy.optimize import linear_sum_assignment

## 2. Setup

In [2]:
#VIDEO_PATH = r"C:\projects\motion-capture-project\data\v01.mp4"
#VIDEO_PATH = r"C:\projects\motion-capture-project\data\scene-01-take-01-cam-01.mp4"
VIDEO_PATH = r"C:\projects\motion-capture-project\data\scene-01-take-01-cam-02.mp4"
# Puedes pasar tiempos como segundos (float/int) o como string:
# "10"        -> segundo 10
# "00:10"     -> minuto 0, segundo 10
# "01:25"     -> minuto 1, segundo 25
# "00:01:25"  -> hora 0, minuto 1, segundo 25
START_TIME = "00:30"
END_TIME = "00:50"

MIN_AREA = 20
MAX_AREA = 2000
MAX_DISTANCE = 50
MAX_MISSING_FRAMES = 30

LOWER_WHITE = np.array([0, 0, 180])
UPPER_WHITE = np.array([180, 80, 255])


## 3. Helpers para usar solo una sección del video


In [3]:
def time_to_seconds(t):
    """
    Convierte tiempo a segundos.
    Acepta:
    - None
    - int/float: segundos
    - "SS"
    - "MM:SS"
    - "HH:MM:SS"
    """
    if t is None:
        return None

    if isinstance(t, (int, float)):
        return float(t)

    if isinstance(t, str):
        parts = t.strip().split(":")
        parts = [float(p) for p in parts]

        if len(parts) == 1:
            return parts[0]
        elif len(parts) == 2:
            minutes, seconds = parts
            return 60 * minutes + seconds
        elif len(parts) == 3:
            hours, minutes, seconds = parts
            return 3600 * hours + 60 * minutes + seconds

    raise ValueError("Formato inválido. Usa segundos, 'MM:SS' o 'HH:MM:SS'.")


def video_info(video_path):
    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        raise FileNotFoundError(f"No se pudo abrir el video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration = total_frames / fps if fps > 0 else None

    cap.release()

    return {
        "fps": fps,
        "total_frames": total_frames,
        "duration_seconds": duration
    }


def frame_range_from_times(video_path, start_time=None, end_time=None):
    info = video_info(video_path)
    fps = info["fps"]
    total_frames = info["total_frames"]
    duration = info["duration_seconds"]

    start_sec = time_to_seconds(start_time)
    end_sec = time_to_seconds(end_time)

    if start_sec is None:
        start_sec = 0.0

    if end_sec is None:
        end_sec = duration

    if start_sec < 0:
        raise ValueError("start_time no puede ser negativo.")

    if end_sec <= start_sec:
        raise ValueError("end_time debe ser mayor que start_time.")

    start_frame = int(round(start_sec * fps))
    end_frame = int(round(end_sec * fps))

    start_frame = max(0, min(start_frame, total_frames))
    end_frame = max(0, min(end_frame, total_frames))

    return start_frame, end_frame, info


## 3. Function to detect markers

In [4]:
def detect_markers(frame):
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    mask = cv2.inRange(hsv, LOWER_WHITE, UPPER_WHITE)

    kernel = np.ones((5, 5), np.uint8)

    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(mask)

    detections = []

    for label in range(1, num_labels):
        x, y, w, h, area = stats[label]

        if area < MIN_AREA:
            continue

        if area > MAX_AREA:
            continue

        cx, cy = centroids[label]
        detections.append([cx, cy])

    return np.array(detections, dtype=np.float32), mask

## 4. Test detection on the first frame

In [5]:
cap = cv2.VideoCapture(VIDEO_PATH)

start_frame, end_frame, info = frame_range_from_times(VIDEO_PATH, START_TIME, END_TIME)

cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
ok, frame = cap.read()
cap.release()

if not ok:
    raise RuntimeError("No se pudo leer el frame inicial de la sección.")

detections, mask = detect_markers(frame)

print("FPS:", info["fps"])
print("Total frames:", info["total_frames"])
print("Duración aprox. del video:", info["duration_seconds"], "segundos")
print("Start frame:", start_frame)
print("End frame:", end_frame)
print("Frames a procesar:", end_frame - start_frame)
print("Detecciones:", len(detections))
print(detections[:10])


FPS: 29.599269692399833
Total frames: 6568
Duración aprox. del video: 221.89736666666667 segundos
Start frame: 888
End frame: 1480
Frames a procesar: 592
Detecciones: 39
[[212.      206.     ]
 [117.15476 253.96428]
 [216.      257.     ]
 [155.      260.57144]
 [203.21083 274.05414]
 [545.5956  273.03677]
 [435.0567  296.49054]
 [287.      303.5    ]
 [420.54718 329.07547]
 [447.      329.     ]]


## 5. View detections

In [7]:
debug = frame.copy()

for x, y in detections:
    cv2.circle(debug, (int(x), int(y)), 6, (0, 0, 255), 2)

cv2.imwrite("debug_detections_frame0.png", debug)
cv2.imwrite("debug_mask_frame0.png", mask)

print("Saved: debug_detections_frame0.png")
print("Saved: debug_mask_frame0.png")

Saved: debug_detections_frame0.png
Saved: debug_mask_frame0.png


## 6. Create Kalman

In [8]:
def create_kalman(x, y):
    kf = cv2.KalmanFilter(4, 2)

    kf.transitionMatrix = np.array([
        [1, 0, 1, 0],
        [0, 1, 0, 1],
        [0, 0, 1, 0],
        [0, 0, 0, 1],
    ], dtype=np.float32)

    kf.measurementMatrix = np.array([
        [1, 0, 0, 0],
        [0, 1, 0, 0],
    ], dtype=np.float32)

    kf.processNoiseCov = np.eye(4, dtype=np.float32) * 0.03
    kf.measurementNoiseCov = np.eye(2, dtype=np.float32) * 1.0
    kf.errorCovPost = np.eye(4, dtype=np.float32)

    kf.statePost = np.array([[x], [y], [0], [0]], dtype=np.float32)

    return kf

## 7. Tracking de una sección del video


In [9]:
def track_video_section(
    video_path,
    start_time=None,
    end_time=None,
    max_missing_frames=MAX_MISSING_FRAMES,
    max_distance=MAX_DISTANCE,
    verbose=True
):
    """
    Procesa solo una sección del video y devuelve:
    - trajectories: array con shape (num_frames_segmento, num_tracks, 2)
    - tracks: lista interna de tracks
    - metadata: información del video y rango usado

    Nota:
    - start_time y end_time pueden ser segundos o strings tipo "MM:SS".
    - Las coordenadas son 2D: [x, y].
    - Los gaps se guardan como [nan, nan].
    """
    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        raise FileNotFoundError(f"No se pudo abrir el video: {video_path}")

    start_frame, end_frame, info = frame_range_from_times(video_path, start_time, end_time)

    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

    tracks = []
    local_frame_idx = 0
    absolute_frame_idx = start_frame

    while absolute_frame_idx < end_frame:
        ok, frame = cap.read()

        if not ok:
            break

        detections, mask = detect_markers(frame)

        if local_frame_idx == 0:
            for x, y in detections:
                tracks.append({
                    "kalman": create_kalman(x, y),
                    "history": [[x, y]],
                    "missing": 0,
                    "active": True
                })

        else:
            predictions = []

            for track in tracks:
                pred = track["kalman"].predict()
                predictions.append([pred[0, 0], pred[1, 0]])

            predictions = np.array(predictions, dtype=np.float32)

            if len(detections) > 0 and len(predictions) > 0:
                cost = np.zeros((len(predictions), len(detections)), dtype=np.float32)

                for i, pred in enumerate(predictions):
                    for j, det in enumerate(detections):
                        cost[i, j] = np.linalg.norm(pred - det)

                rows, cols = linear_sum_assignment(cost)

                assigned_tracks = set()
                assigned_detections = set()

                for r, c in zip(rows, cols):
                    if tracks[r]["missing"] > max_missing_frames:
                        continue

                    if cost[r, c] > max_distance:
                        continue

                    x, y = detections[c]

                    measurement = np.array([[x], [y]], dtype=np.float32)
                    tracks[r]["kalman"].correct(measurement)

                    tracks[r]["history"].append([x, y])
                    tracks[r]["missing"] = 0
                    tracks[r]["active"] = True

                    assigned_tracks.add(r)
                    assigned_detections.add(c)

                for i, track in enumerate(tracks):
                    if i not in assigned_tracks:
                        if track["missing"] <= max_missing_frames:
                            track["history"].append([np.nan, np.nan])
                            track["missing"] += 1
                            track["active"] = False
                        else:
                            track["history"].append([np.nan, np.nan])

                for j, det in enumerate(detections):
                    if j not in assigned_detections:
                        x, y = det

                        tracks.append({
                            "kalman": create_kalman(x, y),
                            "history": [[np.nan, np.nan]] * local_frame_idx + [[x, y]],
                            "missing": 0,
                            "active": True
                        })

            else:
                for track in tracks:
                    if track["missing"] <= max_missing_frames:
                        track["history"].append([np.nan, np.nan])
                        track["missing"] += 1
                        track["active"] = False
                    else:
                        track["history"].append([np.nan, np.nan])

        local_frame_idx += 1
        absolute_frame_idx += 1

    cap.release()

    if len(tracks) == 0:
        trajectories = np.empty((local_frame_idx, 0, 2), dtype=np.float32)
    else:
        max_len = max(len(t["history"]) for t in tracks)

        for t in tracks:
            while len(t["history"]) < max_len:
                t["history"].append([np.nan, np.nan])

        trajectories = np.array([t["history"] for t in tracks], dtype=np.float32)
        trajectories = np.transpose(trajectories, (1, 0, 2))

    metadata = {
        "video_path": video_path,
        "fps": info["fps"],
        "total_video_frames": info["total_frames"],
        "video_duration_seconds": info["duration_seconds"],
        "start_time": start_time,
        "end_time": end_time,
        "start_frame": start_frame,
        "end_frame": end_frame,
        "processed_frames": local_frame_idx,
        "tracks_found": len(tracks)
    }

    if verbose:
        print("FPS:", metadata["fps"])
        print("Start frame:", metadata["start_frame"])
        print("End frame:", metadata["end_frame"])
        print("Frames procesados:", metadata["processed_frames"])
        print("Tracks encontrados:", metadata["tracks_found"])
        print("Trajectories shape:", trajectories.shape)

    return trajectories, tracks, metadata


In [10]:
# Ejemplo:
# Procesar solo desde 00:10 hasta 00:30 del video.
trajectories, tracks, metadata = track_video_section(
    VIDEO_PATH,
    start_time=START_TIME,
    end_time=END_TIME
)


FPS: 29.599269692399833
Start frame: 888
End frame: 1480
Frames procesados: 592
Tracks encontrados: 236
Trajectories shape: (592, 236, 2)


## 8. Revisar resultado


In [11]:
print("Shape:", trajectories.shape)
print("Metadata:")
for key, value in metadata.items():
    print(f"{key}: {value}")


Shape: (592, 236, 2)
Metadata:
video_path: C:\projects\motion-capture-project\data\scene-01-take-01-cam-02.mp4
fps: 29.599269692399833
total_video_frames: 6568
video_duration_seconds: 221.89736666666667
start_time: 00:30
end_time: 00:50
start_frame: 888
end_frame: 1480
processed_frames: 592
tracks_found: 236


## 9. Save trajectories


In [12]:
# Nombre de salida sugerido para no sobrescribir resultados de otras secciones.
output_path = "trajectories_2d_segment.npy"

np.save(output_path, trajectories)

print("Guardado:", output_path)


Guardado: trajectories_2d_segment.npy


## 10. Review a trajectory

In [13]:
frame_id = 0
marker_id = 0

print(trajectories[frame_id, marker_id])

[212. 206.]
